# SECPI Interactive Explorer
### Synergistic & Equitable Cooling Performance Index
Use sliders to adjust parameters and re-run cells to see updated results.

In [1]:
# Enable interactive matplotlib (zoom, pan, resize)
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Patch
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import warnings
import sys
import os

warnings.filterwarnings('ignore')

# Ensure the current directory is in the path so we can import secpi_main
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Import all classes from your main file
from secpi_main import (
    TwoLevelUrbanGrid,
    CorrectedCoolingModel,
    VariableRandomnessACO,
    TreeSpecies,
    SensitivityAnalyzer
)

print("All modules loaded successfully.")
print(f"Working directory: {os.getcwd()}")

ImportError: cannot import name 'VariableRandomnessACO' from 'secpi_main' (c:\Users\Administrator\Downloads\SECPI\secpi_main.py)

---
## 1. Species Overview
Verify that species parameters match the manuscript Table 1.

In [ ]:
ts = TreeSpecies()

# Build a clean comparison table
print(f"{'Species':<18} {'CD (m)':>7} {'Height (m)':>11} {'CPA (m²)':>9} "
      f"{'LAI':>7} {'DBH_cm':>8} {'l0':>6} {'l1':>6} {'h0':>7} {'h1':>6}")
print("-" * 105)

for sp in ts.species_list:
    d = ts.SPECIES_DATA[sp]
    print(f"{sp:<18} {d['crown_diameter_m']:>7.1f} {d['height_m']:>11.1f} "
          f"{d['CPA']:>9.1f} {d['LAI']:>7.3f} {d['DBH_est_cm']:>8.2f} "
          f"{d['l0']:>6.2f} {d['l1']:>6.2f} {d['h0']:>7.1f} {d['h1']:>6.2f}")

print(f"\nMax CPA: {ts.max_CPA:.1f} m²  |  Max LAI: {ts.max_LAI:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

species_names = ts.species_list
cpas = [ts.SPECIES_DATA[sp]['CPA'] for sp in species_names]
lais = [ts.SPECIES_DATA[sp]['LAI'] for sp in species_names]
colors = [ts.SPECIES_DATA[sp]['color'] for sp in species_names]

# Assign shade/evap weights temporarily for cooling potential calculation
ts.shade_weight = 0.7
ts.evap_weight = 0.3
cooling_pots = [ts.get_normalized_cooling_potential(sp) for sp in species_names]

axes[0].barh(species_names, cpas, color=colors, edgecolor='black')
axes[0].set_xlabel('CPA (m²)')
axes[0].set_title('Crown Projection Area')

axes[1].barh(species_names, lais, color=colors, edgecolor='black')
axes[1].set_xlabel('LAI')
axes[1].set_title('Leaf Area Index')

axes[2].barh(species_names, cooling_pots, color=colors, edgecolor='black')
axes[2].set_xlabel('Normalized Score')
axes[2].set_title('Cooling Potential (0.7·CPA + 0.3·LAI)')

plt.tight_layout()
plt.show()

---
## 2. Interactive Urban Grid Generation
Adjust the CA parameters below and click **Generate Grid** to see the urban layout change.

In [ ]:
# --- Widgets ---
w_morphology = widgets.Dropdown(
    options=['organic', 'linear'], value='organic',
    description='Morphology:', style={'description_width': '100px'})
w_p_init = widgets.FloatSlider(
    value=0.15, min=0.05, max=0.45, step=0.05,
    description='p_init:', style={'description_width': '100px'},
    continuous_update=False)
w_alpha = widgets.FloatSlider(
    value=0.1, min=0.0, max=0.5, step=0.05,
    description='CA α:', style={'description_width': '100px'},
    continuous_update=False)
w_beta = widgets.FloatSlider(
    value=0.4, min=0.1, max=0.9, step=0.05,
    description='CA β:', style={'description_width': '100px'},
    continuous_update=False)
w_seed = widgets.IntSlider(
    value=42, min=1, max=999, step=1,
    description='Random Seed:', style={'description_width': '100px'})

btn_generate = widgets.Button(
    description='Generate Grid', button_style='primary',
    icon='refresh', layout=widgets.Layout(width='200px'))

output_grid = widgets.Output()

# Store grid globally so later cells can access it
grid_store = {'grid': None}

def generate_grid(btn):
    with output_grid:
        clear_output(wait=True)
        np.random.seed(w_seed.value)

        grid = TwoLevelUrbanGrid(
            coarse_width=10, coarse_height=10,
            coarse_cell_size=10.0, fine_cell_size=1.0
        )
        grid.generate_ca_archetype(
            params={
                'p_init': w_p_init.value,
                'alpha': w_alpha.value,
                'beta': w_beta.value,
                'theta': 3
            },
            morphology=w_morphology.value
        )
        grid_store['grid'] = grid

        # Visualization
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # Left: Coarse grid
        color_map = {1: [0.6, 0.6, 0.6], 3: [0.6, 0.9, 0.6], 4: [0.95, 0.5, 0.5]}
        display_grid = np.ones((*grid.coarse_grid.shape, 3))
        for val, color in color_map.items():
            display_grid[grid.coarse_grid == val] = color

        ax1.imshow(display_grid, origin='lower', extent=[0, 100, 0, 100])
        ax1.set_title(f'Coarse Grid ({w_morphology.value})', fontsize=12)
        ax1.set_xlabel('X (m)')
        ax1.set_ylabel('Y (m)')

        legend_patches = [
            Patch(facecolor=[0.6, 0.6, 0.6], label=f'Prohibited (Building)'),
            Patch(facecolor=[0.6, 0.9, 0.6], label=f'Available (Plantable)'),
            Patch(facecolor=[0.95, 0.5, 0.5], label=f'Vulnerable Zone'),
        ]
        ax1.legend(handles=legend_patches, loc='upper right', fontsize=8)

        # Right: Statistics
        unique, counts = np.unique(grid.coarse_grid, return_counts=True)
        stats = dict(zip(unique, counts))
        total = sum(counts)

        labels = ['Prohibited\n(Building)', 'Available\n(Plantable)', 'Vulnerable\nZone']
        values = [stats.get(1, 0), stats.get(3, 0), stats.get(4, 0)]
        bar_colors = ['gray', 'lightgreen', 'salmon']

        bars = ax2.bar(labels, values, color=bar_colors, edgecolor='black')
        for bar, val in zip(bars, values):
            pct = val / total * 100
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                     f'{val}\n({pct:.0f}%)', ha='center', va='bottom',
                     fontweight='bold', fontsize=10)
        ax2.set_ylabel('Number of Coarse Cells')
        ax2.set_title('Land Use Distribution')
        ax2.set_ylim(0, max(values) * 1.3)

        plt.tight_layout()
        plt.show()

        print(f"\n✓ Grid generated. {len(grid.plantable_coords)} plantable cells available.")

btn_generate.on_click(generate_grid)

controls = widgets.VBox([
    widgets.HBox([w_morphology, w_seed]),
    widgets.HBox([w_p_init, w_alpha, w_beta]),
    btn_generate
])

display(controls, output_grid)

---
## 3. Single Tree Cooling Decay
Visualize how cooling intensity decays with distance for each species.
Select a species to see its radial decay profile.

In [ ]:
w_species = widgets.Dropdown(
    options=ts.species_list, value='Narra',
    description='Species:', style={'description_width': '80px'})
w_lambda = widgets.FloatSlider(
    value=0.1, min=0.01, max=0.5, step=0.01,
    description='λ (decay):', style={'description_width': '80px'},
    continuous_update=False)

output_decay = widgets.Output()

def plot_decay(change):
    with output_decay:
        clear_output(wait=True)

        grid = grid_store.get('grid')
        if grid is None:
            print("⚠ Generate a grid first (Section 2 above).")
            return

        cooling_model = CorrectedCoolingModel(
            decay_lambda=w_lambda.value,
            cca_threshold=1.2,
            competition_k=5.0
        )

        species_name = w_species.value
        center = (grid.fine_width / 2, grid.fine_height / 2)

        cooling = cooling_model.calculate_cooling_contribution(
            center, species_name, grid.fine_grid_points
        )
        distances = np.sqrt(
            (grid.fine_grid_points[:, 0] - center[0])**2 +
            (grid.fine_grid_points[:, 1] - center[1])**2
        )

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

        # Left: Heatmap
        cg = cooling.reshape(grid.n_rows_fine, grid.n_cols_fine)
        im = ax1.imshow(cg.T, extent=[0, grid.fine_width, 0, grid.fine_height],
                        origin='lower', cmap='coolwarm_r', aspect='equal')
        ax1.scatter(*center, color='red', s=120, marker='x', linewidths=2,
                    zorder=5, label='Tree')

        sp_data = ts.SPECIES_DATA[species_name]
        crown_r = sp_data['crown_diameter_m'] / 2
        circle = Circle(center, crown_r, fill=False, color='red',
                        linewidth=2, linestyle='--', label=f'Crown (r={crown_r:.1f}m)')
        ax1.add_patch(circle)
        ax1.set_title(f'Cooling Heatmap: {species_name}')
        ax1.set_xlabel('X (m)')
        ax1.set_ylabel('Y (m)')
        ax1.legend(loc='upper left', fontsize=8)
        plt.colorbar(im, ax=ax1, label='Cooling Intensity', shrink=0.8)

        # Right: Radial profile
        sort_idx = np.argsort(distances)
        ax2.scatter(distances[sort_idx], cooling[sort_idx],
                    s=1, alpha=0.3, color='steelblue')

        # Smooth line
        bins = np.linspace(0, distances.max(), 100)
        bin_means = []
        for i in range(len(bins) - 1):
            mask = (distances >= bins[i]) & (distances < bins[i+1])
            if np.any(mask):
                bin_means.append((bins[i] + bins[i+1]) / 2)
            else:
                bin_means.append(np.nan)
        bin_centers = np.array(bin_means)

        bin_vals = []
        for i in range(len(bins) - 1):
            mask = (distances >= bins[i]) & (distances < bins[i+1])
            if np.any(mask):
                bin_vals.append(np.mean(cooling[mask]))
            else:
                bin_vals.append(np.nan)
        bin_vals = np.array(bin_vals)

        valid = ~np.isnan(bin_centers) & ~np.isnan(bin_vals)
        ax2.plot(bin_centers[valid], bin_vals[valid], 'b-', linewidth=2.5,
                 label='Mean decay')

        ax2.axvline(crown_r, color='red', linestyle='--', linewidth=1.5,
                    label=f'Crown radius ({crown_r:.1f}m)')
        ax2.set_xlabel('Distance from Tree (m)')
        ax2.set_ylabel('Cooling Intensity')
        ax2.set_title(f'Radial Decay (λ={w_lambda.value})')
        ax2.legend(fontsize=8)
        ax2.grid(True, alpha=0.3)

        # Info box
        ax2.text(0.95, 0.95,
                 f"CPA: {sp_data['CPA']:.1f} m²\n"
                 f"LAI: {sp_data['LAI']:.3f}\n"
                 f"CD: {sp_data['crown_diameter_m']:.1f} m\n"
                 f"Height: {sp_data['height_m']:.1f} m",
                 transform=ax2.transAxes, fontsize=9,
                 verticalalignment='top', horizontalalignment='right',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

        plt.tight_layout()
        plt.show()

w_species.observe(plot_decay, names='value')
w_lambda.observe(plot_decay, names='value')

display(widgets.HBox([w_species, w_lambda]), output_decay)

# Trigger initial plot
plot_decay(None)

---
## 4. Compare All Species Decay Profiles
Side-by-side comparison at a fixed λ value.

In [ ]:
def plot_all_species_comparison(decay_lambda=0.1):
    grid = grid_store.get('grid')
    if grid is None:
        print("⚠ Generate a grid first.")
        return

    cooling_model = CorrectedCoolingModel(decay_lambda=decay_lambda)
    center = (grid.fine_width / 2, grid.fine_height / 2)

    fig, ax = plt.subplots(figsize=(10, 6))

    for sp in ts.species_list:
        cooling = cooling_model.calculate_cooling_contribution(
            center, sp, grid.fine_grid_points)
        distances = np.sqrt(
            (grid.fine_grid_points[:, 0] - center[0])**2 +
            (grid.fine_grid_points[:, 1] - center[1])**2
        )

        # Binned averages for clean lines
        max_d = 50  # plot up to 50m
        bins = np.linspace(0, max_d, 80)
        bin_vals = []
        bin_centers = []
        for i in range(len(bins) - 1):
            mask = (distances >= bins[i]) & (distances < bins[i+1])
            if np.any(mask):
                bin_vals.append(np.mean(cooling[mask]))
                bin_centers.append((bins[i] + bins[i+1]) / 2)

        color = ts.SPECIES_DATA[sp]['color']
        crown_r = ts.SPECIES_DATA[sp]['crown_diameter_m'] / 2
        ax.plot(bin_centers, bin_vals, color=color, linewidth=2.5,
                label=f'{sp} (CD={ts.SPECIES_DATA[sp]["crown_diameter_m"]:.0f}m)')

        # Mark crown radius
        ax.axvline(crown_r, color=color, linestyle=':', alpha=0.4, linewidth=1)

    ax.set_xlabel('Distance from Tree (m)', fontsize=11)
    ax.set_ylabel('Cooling Intensity', fontsize=11)
    ax.set_title(f'Species Comparison: Radial Cooling Decay (λ={decay_lambda})',
                 fontsize=13)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 50)
    plt.tight_layout()
    plt.show()

plot_all_species_comparison(0.1)

---
## 5. Run ACO Optimization
Configure parameters and run the optimizer. Results update below.

In [ ]:
# --- ACO Widgets ---
w_n_trees = widgets.IntSlider(
    value=5, min=1, max=15, description='Trees:',
    style={'description_width': '100px'}, continuous_update=False)
w_n_species_aco = widgets.IntSlider(
    value=6, min=1, max=6, description='Species Pool:',
    style={'description_width': '100px'}, continuous_update=False)
w_n_ants = widgets.IntSlider(
    value=15, min=5, max=40, step=5, description='Ants:',
    style={'description_width': '100px'}, continuous_update=False)
w_n_iter = widgets.IntSlider(
    value=30, min=5, max=80, step=5, description='Iterations:',
    style={'description_width': '100px'}, continuous_update=False)
w_q0 = widgets.FloatSlider(
    value=0.7, min=0.1, max=1.0, step=0.05, description='q₀ (exploit):',
    style={'description_width': '100px'}, continuous_update=False)
w_decay = widgets.FloatSlider(
    value=0.1, min=0.01, max=0.5, step=0.01, description='λ (decay):',
    style={'description_width': '100px'}, continuous_update=False)
w_cca_thresh = widgets.FloatSlider(
    value=1.2, min=0.5, max=3.0, step=0.1, description='CCA Thresh:',
    style={'description_width': '100px'}, continuous_update=False)

btn_run_aco = widgets.Button(
    description='Run ACO Optimization', button_style='success',
    icon='play', layout=widgets.Layout(width='250px', height='40px'))

progress_bar = widgets.IntProgress(
    value=0, min=0, max=100, description='Progress:',
    bar_style='info', layout=widgets.Layout(width='400px'))

output_aco = widgets.Output()

# Store ACO results globally
aco_store = {'aco': None, 'cooling_model': None}

def run_aco(btn):
    with output_aco:
        clear_output(wait=True)

        grid = grid_store.get('grid')
        if grid is None:
            print("⚠ Generate a grid first (Section 2).")
            return

        if len(grid.plantable_coords) == 0:
            print("⚠ No plantable cells. Regenerate grid with lower p_init.")
            return

        progress_bar.value = 10
        print("Building cooling model...")

        cooling_model = CorrectedCoolingModel(
            decay_lambda=w_decay.value,
            cca_threshold=w_cca_thresh.value,
            competition_k=5.0
        )
        aco_store['cooling_model'] = cooling_model

        progress_bar.value = 20
        print("Initializing ACO...")

        aco = VariableRandomnessACO(
            grid, cooling_model,
            n_trees=w_n_trees.value,
            n_ants=w_n_ants.value,
            n_iterations=w_n_iter.value,
            evaporation_rate=0.5,
            alpha=1.0, beta=2.0,
            q0=w_q0.value,
            random_seed=None,
            n_species_restricted=w_n_species_aco.value
        )

        progress_bar.value = 30
        print(f"Running ACO ({w_n_ants.value} ants × {w_n_iter.value} iterations)...\n")

        history_best, history_avg = aco.run(verbose=True)
        aco_store['aco'] = aco

        progress_bar.value = 80
        print("\nGenerating visualizations...")

        # --- VISUALIZATION ---
        fig = plt.figure(figsize=(16, 12))
        gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

        # Panel 1: Convergence
        ax_conv = fig.add_subplot(gs[0, 0])
        ax_conv.plot(history_best, 'b-', linewidth=2, label='Best SECPI')
        ax_conv.plot(history_avg, 'r--', linewidth=1.5, alpha=0.7, label='Avg SECPI')
        ax_conv.set_xlabel('Iteration')
        ax_conv.set_ylabel('SECPI')
        ax_conv.set_title('ACO Convergence')
        ax_conv.legend(fontsize=9)
        ax_conv.grid(True, alpha=0.3)

        if aco.best_solution:
            tree_coords, tree_species = aco.best_solution
            display_cooling, cca_vals = cooling_model.calculate_total_cooling(
                tree_coords, tree_species, grid.fine_grid_points,
                apply_competition=True)

            # Panel 2: Tree placements
            ax_place = fig.add_subplot(gs[0, 1])
            for i in range(grid.coarse_height):
                for j in range(grid.coarse_width):
                    x = j * grid.coarse_cell_size
                    y = i * grid.coarse_cell_size
                    lu = grid.coarse_grid[i, j]
                    cmap = {1: 'gray', 3: 'lightgreen', 4: 'salmon'}
                    rect = Rectangle(
                        (x, y), grid.coarse_cell_size, grid.coarse_cell_size,
                        facecolor=cmap.get(lu, 'white'),
                        edgecolor='black', linewidth=0.3, alpha=0.6)
                    ax_place.add_patch(rect)

            tree_species_obj = TreeSpecies()
            for (tx, ty), sp in zip(tree_coords, tree_species):
                color = tree_species_obj.get_species_color(sp)
                ax_place.scatter(tx, ty, color=color, s=120,
                                 edgecolors='black', linewidth=1.5, zorder=5)
                cr = tree_species_obj.get_crown_radius(sp)
                circle = Circle((tx, ty), cr, color=color,
                                alpha=0.15, linewidth=1)
                ax_place.add_patch(circle)

            ax_place.set_xlim(0, grid.fine_width)
            ax_place.set_ylim(0, grid.fine_height)
            ax_place.set_aspect('equal')
            ax_place.set_title('Tree Placements')

            # Panel 3: Cooling heatmap
            ax_heat = fig.add_subplot(gs[1, 0])
            cg = display_cooling.reshape(grid.n_rows_fine, grid.n_cols_fine)
            im = ax_heat.imshow(cg.T,
                                extent=[0, grid.fine_width, 0, grid.fine_height],
                                origin='lower', cmap='coolwarm_r', aspect='equal')
            for (tx, ty), sp in zip(tree_coords, tree_species):
                ax_heat.scatter(tx, ty, color='white', s=50,
                                edgecolors='black', linewidth=1, zorder=5)
            ax_heat.set_title('Cooling Distribution')
            plt.colorbar(im, ax=ax_heat, label='Cooling Intensity', shrink=0.8)

            # Panel 4: Results summary
            ax_text = fig.add_subplot(gs[1, 1])
            ax_text.axis('off')

            species_count = {}
            for sp in tree_species:
                species_count[sp] = species_count.get(sp, 0) + 1

            summary_lines = [
                f"SECPI Score: {aco.best_secpi:.4f}",
                f"Trees Placed: {len(tree_coords)}",
                f"Species Pool: {w_n_species_aco.value}",
                "",
                "Species Used:",
            ]
            for sp, count in species_count.items():
                cr = tree_species_obj.get_crown_radius(sp)
                summary_lines.append(f"  • {sp} ×{count} (r={cr:.1f}m)")

            summary_lines.extend([
                "",
                "Cooling Statistics:",
                f"  Mean: {np.mean(display_cooling):.4f}",
                f"  Max:  {np.max(display_cooling):.4f}",
                f"  Std:  {np.std(display_cooling):.4f}",
                "",
                f"Parameters:",
                f"  λ={w_decay.value}, CCA_thresh={w_cca_thresh.value}",
                f"  q₀={w_q0.value}, Ants={w_n_ants.value}, Iter={w_n_iter.value}",
            ])

            ax_text.text(0.05, 0.95, "\n".join(summary_lines),
                         transform=ax_text.transAxes, fontsize=11,
                         verticalalignment='top', fontfamily='monospace',
                         bbox=dict(boxstyle='round', facecolor='lightyellow',
                                   alpha=0.8))
            ax_text.set_title('Results Summary')

        plt.suptitle(f'ACO Optimization Results (SECPI = {aco.best_secpi:.4f})',
                     fontsize=14, fontweight='bold')
        plt.show()

        progress_bar.value = 100
        print(f"\n✓ Optimization complete. Best SECPI: {aco.best_secpi:.4f}")

btn_run_aco.on_click(run_aco)

# Layout
row1 = widgets.HBox([w_n_trees, w_n_species_aco, w_q0])
row2 = widgets.HBox([w_n_ants, w_n_iter])
row3 = widgets.HBox([w_decay, w_cca_thresh])

display(widgets.VBox([row1, row2, row3, widgets.HBox([btn_run_aco, progress_bar])]))
display(output_aco)

---
## 6. Equity Analysis
Examine cooling distribution across Prohibited, Available, and Vulnerable zones.

In [ ]:
def run_equity_analysis():
    grid = grid_store.get('grid')
    aco = aco_store.get('aco')

    if grid is None or aco is None or aco.best_cooling is None:
        print("⚠ Run grid generation (Section 2) and ACO (Section 5) first.")
        return

    cooling = aco.best_cooling

    # Map coarse land use to fine grid
    scale = int(grid.coarse_cell_size / grid.fine_cell_size)
    fine_lu = np.repeat(np.repeat(grid.coarse_grid, scale, axis=0),
                        scale, axis=1).flatten()

    zones = {
        'Vulnerable (V)': fine_lu == 4,
        'Available (A)': fine_lu == 3,
        'Building (P)': fine_lu == 1,
    }

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Box plot
    box_data = []
    box_labels = []
    box_colors = ['salmon', 'lightgreen', 'gray']
    for (name, mask), color in zip(zones.items(), box_colors):
        if np.any(mask):
            box_data.append(cooling[mask])
            box_labels.append(f"{name}\n(n={np.sum(mask):,})")

    bp = ax1.boxplot(box_data, labels=box_labels, patch_artist=True,
                     showfliers=False, widths=0.6)
    for patch, color in zip(bp['boxes'], box_colors[:len(box_data)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax1.set_ylabel('Cooling Intensity')
    ax1.set_title('Cooling Distribution by Zone')
    ax1.grid(axis='y', alpha=0.3)

    # Right: Bar chart of means with global reference
    zone_means = {}
    for name, mask in zones.items():
        if np.any(mask):
            zone_means[name] = np.mean(cooling[mask])

    global_mean = np.mean(cooling)

    bars = ax2.bar(zone_means.keys(), zone_means.values(),
                   color=box_colors[:len(zone_means)],
                   edgecolor='black', alpha=0.7)
    ax2.axhline(global_mean, color='navy', linestyle='--', linewidth=2,
                label=f'Global Mean ({global_mean:.4f})')

    for bar, val in zip(bars, zone_means.values()):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

    # Equity ratio
    vuln_mean = zone_means.get('Vulnerable (V)', 0)
    equity_ratio = vuln_mean / global_mean if global_mean > 0 else 0

    ax2.set_ylabel('Mean Cooling Intensity')
    ax2.set_title(f'Zonal Mean Cooling (Equity Ratio: {equity_ratio:.2f})')
    ax2.legend(fontsize=10)
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Interpretation
    print(f"\n{'='*50}")
    print(f"EQUITY ANALYSIS")
    print(f"{'='*50}")
    print(f"  Global mean cooling:     {global_mean:.4f}")
    print(f"  Vulnerable zone mean:    {vuln_mean:.4f}")
    print(f"  Equity ratio (V/Global): {equity_ratio:.2f}")
    if equity_ratio > 1.2:
        print(f"  → ✓ HIGH EQUITY: Vulnerable zones well-served.")
    elif equity_ratio > 1.0:
        print(f"  → ~ MODERATE: Vulnerable zones at average level.")
    else:
        print(f"  → ✗ LOW EQUITY: Vulnerable zones underserved.")

run_equity_analysis()

---
## 7. Quick Sensitivity Check
Compare SECPI across different λ values to see which parameters matter most.

In [ ]:
def quick_sensitivity_sweep():
    grid = grid_store.get('grid')
    if grid is None:
        print("⚠ Generate a grid first.")
        return

    print("Running quick sensitivity sweep (this takes a few minutes)...\n")

    # Sweep decay_lambda
    lambdas = [0.03, 0.05, 0.1, 0.15, 0.25, 0.4]
    secpi_by_lambda = []

    for lam in lambdas:
        cm = CorrectedCoolingModel(decay_lambda=lam)
        scores = []
        for trial in range(3):
            try:
                aco = VariableRandomnessACO(
                    grid, cm, n_trees=5, n_ants=8,
                    n_iterations=10, q0=0.7)
                aco.run(verbose=False)
                scores.append(aco.best_secpi)
            except:
                pass
        mean_s = np.mean(scores) if scores else np.nan
        secpi_by_lambda.append(mean_s)
        print(f"  λ={lam:.2f} → SECPI={mean_s:.4f}")

    # Sweep shade_weight
    shade_weights = [0.3, 0.5, 0.6, 0.7, 0.8, 0.9]
    secpi_by_sw = []

    for sw in shade_weights:
        cm = CorrectedCoolingModel(
            decay_lambda=0.1, shade_weight=sw, evap_weight=1.0 - sw)
        scores = []
        for trial in range(3):
            try:
                aco = VariableRandomnessACO(
                    grid, cm, n_trees=5, n_ants=8,
                    n_iterations=10, q0=0.7)
                aco.run(verbose=False)
                scores.append(aco.best_secpi)
            except:
                pass
        mean_s = np.mean(scores) if scores else np.nan
        secpi_by_sw.append(mean_s)
        print(f"  shade_weight={sw:.1f} → SECPI={mean_s:.4f}")

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.plot(lambdas, secpi_by_lambda, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Decay λ', fontsize=11)
    ax1.set_ylabel('SECPI', fontsize=11)
    ax1.set_title('Sensitivity to Decay Rate')
    ax1.grid(True, alpha=0.3)

    ax2.plot(shade_weights, secpi_by_sw, 'go-', linewidth=2, markersize=8)
    ax2.set_xlabel('Shade Weight (CPA)', fontsize=11)
    ax2.set_ylabel('SECPI', fontsize=11)
    ax2.set_title('Sensitivity to Shade vs ET Weight')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

quick_sensitivity_sweep()

---
## 8. Species Portfolio Comparison
How does SECPI change as you add more species to the available pool?

In [ ]:
def species_portfolio_comparison():
    grid = grid_store.get('grid')
    if grid is None:
        print("⚠ Generate a grid first.")
        return

    print("Comparing species portfolio sizes (1 to 6 species)...\n")

    cm = CorrectedCoolingModel(decay_lambda=0.1)
    results = {}

    for n_sp in range(1, 7):
        scores = []
        for trial in range(3):
            try:
                aco = VariableRandomnessACO(
                    grid, cm, n_trees=5, n_ants=10,
                    n_iterations=15, q0=0.7,
                    n_species_restricted=n_sp)
                aco.run(verbose=False)
                scores.append(aco.best_secpi)

                if trial == 0 and aco.best_solution:
                    results[n_sp] = {
                        'secpi_scores': scores,
                        'best_species': aco.best_solution[1]
                    }
            except:
                pass

        mean_s = np.mean(scores) if scores else 0
        if n_sp in results:
            results[n_sp]['mean_secpi'] = mean_s
            results[n_sp]['std_secpi'] = np.std(scores) if scores else 0
        else:
            results[n_sp] = {'mean_secpi': mean_s, 'std_secpi': 0,
                             'best_species': []}

        species_used = results[n_sp].get('best_species', [])
        print(f"  {n_sp} species → SECPI={mean_s:.4f} "
              f"(species: {species_used})")

    # Plot
    n_species_range = sorted(results.keys())
    means = [results[n]['mean_secpi'] for n in n_species_range]
    stds = [results[n]['std_secpi'] for n in n_species_range]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(n_species_range, means, yerr=stds,
                fmt='o-', linewidth=2, markersize=10,
                capsize=5, color='teal', ecolor='gray')
    ax.set_xlabel('Number of Species in Pool', fontsize=12)
    ax.set_ylabel('Mean SECPI', fontsize=12)
    ax.set_title('Effect of Species Diversity on SECPI', fontsize=13)
    ax.set_xticks(n_species_range)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

species_portfolio_comparison()